# 6장 — 학습 최적화와 DPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch06_training_optimization_dpo.ipynb)

이 노트북은 『밑바닥부터 시작하는 딥러닝 6』의 공식 코드 저장소를 기준으로 구성했습니다. T4에서 실행하기 어렵다는 이유로 알고리즘이나 모델 구조를 토이 버전으로 바꾸지 않습니다.

- 기준 upstream commit: `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`
- 포함한 장 코드 파일 수: **9개**
- 함께 펼쳐서 보여주는 공통 모듈 수: **3개**


## 노트북 구성 원칙

1. 공식 `.py`의 모델 구조와 계산 로직을 그대로 유지합니다.
2. 함수·클래스·실행부를 셀 단위로 나눠 위에서 아래로 읽기 쉽게 배치합니다.
3. 일본어 자연어 주석은 한국어로 바꾸며, 변수명·수식·텐서 shape 같은 기술 표기는 유지합니다.
4. 공통 `codebot` / `storybot` 모듈도 외부 파일 뒤에 숨기지 않고 이 노트북에서 직접 확인할 수 있게 합니다.
5. T4에서 시간이 오래 걸리는 전체 학습 스케줄도 기본값 자체를 임의 축소하지 않습니다.


## 0. Colab 환경 준비

먼저 공식 저장소를 고정된 커밋으로 준비하고 현재 런타임의 GPU를 확인합니다.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('작업 경로:', Path.cwd())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch 확인 중 오류:', exc)


## 1. 이 장에서 사용하는 공통 구현

장 코드가 import하는 로컬 모듈을 먼저 읽습니다. 긴 파일도 클래스·함수 단위로 나눠 표시합니다.


### `storybot/model.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile storybot/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


#### `RoPE` 클래스 구현


In [ ]:
%%writefile -a storybot/model.py


class RoPE(nn.Module):
    def __init__(self, theta, key_dim, max_context_len):
        super().__init__()
        assert key_dim % 2 == 0
        half = key_dim // 2

        half_ids = torch.arange(0, half)
        inv_freq = 1.0 / (theta ** ( (2.0 * half_ids) / key_dim ))  # 출력 예시: (half,)

        positions = torch.arange(max_context_len)  # 출력 예시: (max_context_len,)
        angles = positions[:, None] * inv_freq[None, :]  # 출력 예시: (max_context_len, half)

        cos = torch.cos(angles)  # 출력 예시: (max_context_len, half)
        sin = torch.sin(angles)  # 출력 예시: (max_context_len, half)

        self.register_buffer("cos_cache", cos)
        self.register_buffer("sin_cache", sin)

    def forward(self, x, offset=0):
        batch_size, num_head, context_len, key_dim = x.shape

        # 이 코드 단계의 동작을 확인하는 예시
        input_dtype = x.dtype
        x = x.float()

        # 이 코드 단계의 동작을 확인하는 예시
        max_context_len = self.cos_cache.size(0)
        if offset + context_len > max_context_len:
            offset = max_context_len - context_len

        cos = self.cos_cache[offset:offset + context_len]
        sin = self.sin_cache[offset:offset + context_len]

        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd  = x_even * sin + x_odd * cos

        out = torch.stack([x_rot_even, x_rot_odd], dim=-1)
        out = out.reshape(batch_size, num_head, context_len, key_dim)

        return out.to(input_dtype)  # 이 코드 단계의 동작을 확인하는 예시


#### `MultiHeadAttention` 클래스 구현


In [ ]:
%%writefile -a storybot/model.py

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, rope=None):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.rope = rope

        # 이 코드 단계의 동작을 확인하는 예시
        self.k_cache = None  # Key의캐시
        self.v_cache = None  # Value의캐시
        self.cache_offset = 0  # 이 코드 단계의 동작을 확인하는 예시

    def forward(self, x, use_cache=False):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, C, H, D).transpose(1, 2)
        K = K.view(B, C, H, D).transpose(1, 2)
        V = V.view(B, C, H, D).transpose(1, 2)

        # 이 코드 단계의 동작을 확인하는 예시
        if self.rope is not None:
            if use_cache:
                Q = self.rope(Q, self.cache_offset)
                K = self.rope(K, self.cache_offset)
            else:
                Q = self.rope(Q)
                K = self.rope(K)

        # KV-Cache의처리
        if use_cache:
            # 이 코드 단계의 동작을 확인하는 예시
            is_first_call = (self.k_cache is None)

            if is_first_call:
                # 이 코드 단계의 동작을 확인하는 예시
                self.k_cache = K
                self.v_cache = V
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                self.k_cache = torch.cat([self.k_cache, K], dim=2)
                self.v_cache = torch.cat([self.v_cache, V], dim=2)

            # 이 코드 단계의 동작을 확인하는 예시
            self.cache_offset += C

            # 이 코드 단계의 동작을 확인하는 예시
            K = self.k_cache
            V = self.v_cache

        # 이 코드 단계의 동작을 확인하는 예시
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (D ** 0.5)

        # Causal Mask의적용
        # 이 코드 단계의 동작을 확인하는 예시
        # 이 코드 단계의 동작을 확인하는 예시
        # 이 코드 단계의 동작을 확인하는 예시
        if not use_cache or (use_cache and is_first_call):
            mask = torch.tril(torch.ones(C, C, device=scores.device))
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)

        hidden = hidden.transpose(1, 2).contiguous()
        hidden = hidden.view(B, C, H * D)
        output = self.W_o(hidden)
        return output

    def clear_cache(self):
        """キャッシュをクリアする"""
        self.k_cache = None
        self.v_cache = None
        self.cache_offset = 0


#### `silu()` 함수 구현


In [ ]:
%%writefile -a storybot/model.py

def silu(x):
    return x * torch.sigmoid(x)


#### `SwiGLU` 클래스 구현


In [ ]:
%%writefile -a storybot/model.py

class SwiGLU(nn.Module):
    def __init__(self, x_dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(x_dim * 8 / 3)

        self.W = nn.Linear(x_dim, hidden_dim, bias=False)
        self.V = nn.Linear(x_dim, hidden_dim, bias=False)
        self.O = nn.Linear(hidden_dim, x_dim, bias=False)

    def forward(self, x):
        a = self.W(x)
        b = self.V(x)

        gated = F.silu(a) * b  # 참고: silu(a) * b
        out = self.O(gated)
        return out


#### `Block` 클래스 구현


In [ ]:
%%writefile -a storybot/model.py

class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, rope=None):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.RMSNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, rope)
        self.norm2 = nn.RMSNorm(embed_dim)
        self.ffn = SwiGLU(embed_dim, ff_dim)

    def forward(self, x, use_cache=False):
        x = x + self.attn(self.norm1(x), use_cache=use_cache)
        x = x + self.ffn(self.norm2(x))
        return x

    def clear_cache(self):
        """キャッシュをクリアする"""
        self.attn.clear_cache()


#### `GPT` 클래스 구현


In [ ]:
%%writefile -a storybot/model.py


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, theta=10000):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.theta = theta

        self.embed = nn.Embedding(vocab_size, embed_dim)

        head_dim = embed_dim // n_head
        rope = RoPE(theta, head_dim, max_context_len)

        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, rope)
            for _ in range(n_layer)
        ])

        self.norm = nn.RMSNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids, use_cache=False):
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        x = self.norm(x)
        logits = self.unembed(x)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'theta': self.theta,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            theta=checkpoint['theta']
        )

        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model

    def clear_cache(self):
        for block in self.blocks:
            block.clear_cache()


#### 조건에 따른 실행


In [ ]:
%%writefile -a storybot/model.py

if __name__ == "__main__":
    vocab_size = 1000
    max_context_len = 256
    embed_dim = 384
    n_head = 6
    n_layer = 6
    ff_dim = int(embed_dim * 8 / 3)
    theta = 10000

    # 모델생성
    model = GPT(vocab_size, max_context_len, embed_dim, n_head,
                n_layer, ff_dim, theta)
    # 이 코드 단계의 동작을 확인하는 예시
    dummy_input = torch.randint(0, vocab_size, (1, max_context_len))
    logits = model(dummy_input)
    print(f"出力形状: {logits.shape}")


### `storybot/tokenizer.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile storybot/tokenizer.py
import os
import pickle
from multiprocessing import Pool
import shutil
from collections import defaultdict
import regex as re
from tqdm import tqdm
import numpy as np


#### `pretokenize()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    for m in re.finditer(pattern, text):
        yield m.group(0)


#### `count_pairs()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts


#### `merge()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


#### `find_chunk_boundaries()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def find_chunk_boundaries(file_path, num_chunks, end_token="<|endoftext|>"):
    byte_end_token = end_token.encode("utf-8")

    with open(file_path, "rb") as file:  # 이 코드 단계의 동작을 확인하는 예시
        # 이 코드 단계의 동작을 확인하는 예시
        file.seek(0, os.SEEK_END)
        file_size = file.tell()
        file.seek(0)

        chunk_size = file_size // num_chunks

        # 이 코드 단계의 동작을 확인하는 예시
        chunk_boundaries = [i * chunk_size for i in range(num_chunks)]
        chunk_boundaries.append(file_size)  # 이 코드 단계의 동작을 확인하는 예시

        buffer_size = 4096  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for bi in range(1, len(chunk_boundaries) - 1):
            chunk_position = chunk_boundaries[bi]
            file.seek(chunk_position)  # 이 코드 단계의 동작을 확인하는 예시

            while True:
                buffer = file.read(buffer_size)  # 이 코드 단계의 동작을 확인하는 예시

                # 이 코드 단계의 동작을 확인하는 예시
                if buffer == b"":
                    chunk_boundaries[bi] = file_size
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                end_position = buffer.find(byte_end_token)
                if end_position != -1:
                    # 이 코드 단계의 동작을 확인하는 예시
                    chunk_boundaries[bi] = chunk_position + end_position
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                chunk_position += buffer_size

    # 이 코드 단계의 동작을 확인하는 예시
    return sorted(set(chunk_boundaries))


#### `process_single_chunk()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def process_single_chunk(file_path, start, end, end_token):
    """1つのチャンクを処理する関数"""
    pretoken_counts = defaultdict(int)

    # 이 코드 단계의 동작을 확인하는 예시
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk_byte = f.read(end - start)
        chunk_text = chunk_byte.decode("utf-8", errors="ignore")

        # 이 코드 단계의 동작을 확인하는 예시
        texts = chunk_text.split(end_token)

        # 이 코드 단계의 동작을 확인하는 예시
        for text in texts:
            for pretoken in pretokenize(text):
                pretoken_counts[pretoken] += 1

    return pretoken_counts


#### `pretoken_chunk()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def pretoken_chunk(args):
    file_path, start, end, end_token = args
    pretoken_counts = defaultdict(int)

    # 이 코드 단계의 동작을 확인하는 예시
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk_byte = f.read(end - start)
        chunk_text = chunk_byte.decode("utf-8", errors="ignore")

        # 이 코드 단계의 동작을 확인하는 예시
        texts = chunk_text.split(end_token)

        # 이 코드 단계의 동작을 확인하는 예시
        for text in texts:
            for pretoken in pretokenize(text):
                pretoken_counts[pretoken] += 1

    return pretoken_counts


#### `train_bpe()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def train_bpe(file_path, vocab_size, end_token="<|endoftext|>", num_processes=8, num_chunks=8):
    # 이 코드 단계의 동작을 확인하는 예시
    chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
    total_chunks = len(chunk_boundaries) - 1

    chunk_info_list = []
    for i in range(total_chunks):
        start = chunk_boundaries[i]
        end = chunk_boundaries[i + 1]
        chunk_info_list.append((file_path, start, end, end_token))

    # 이 코드 단계의 동작을 확인하는 예시
    with Pool(processes=num_processes) as pool:
        all_results = list(tqdm(pool.imap(pretoken_chunk, chunk_info_list), total=len(chunk_info_list), desc="Pretokenizing"))

    # 이 코드 단계의 동작을 확인하는 예시
    pretoken_counts = defaultdict(int)
    for chunk_result in all_results:
        for pretoken, count in chunk_result.items():
            pretoken_counts[pretoken] += count

    # 이 코드 단계의 동작을 확인하는 예시
    ids_counts = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counts.items()}


    num_merges = vocab_size - 256 - 1
    merge_rules = {}
    pair_to_ids = defaultdict(set)  # 캐시

    pair_counts = defaultdict(int)
    for ids, count in ids_counts.items():
        count_pairs(ids, count, pair_counts)
        for pair in zip(ids, ids[1:]):  # 이 코드 단계의 동작을 확인하는 예시
            pair_to_ids[pair].add(ids)

    for step in tqdm(range(num_merges), desc="Training BPE"):
        if not pair_counts:  # 이 코드 단계의 동작을 확인하는 예시
            break

        # 이 코드 단계의 동작을 확인하는 예시
        # 참고: best_pair = max(pair_counts, key=pair_counts.get)
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        affected_ids = pair_to_ids[best_pair]
        del pair_to_ids[best_pair]  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for ids in affected_ids:
            ids_count = ids_counts[tuple(ids)]
            new_ids = merge(ids, best_pair, new_id)

            del ids_counts[tuple(ids)]  # 이 코드 단계의 동작을 확인하는 예시
            ids_counts[tuple(new_ids)] = ids_count  # 이 코드 단계의 동작을 확인하는 예시

            # 이 코드 단계의 동작을 확인하는 예시
            old_counts = count_pairs(ids)
            for pair, count in old_counts.items():
                pair_counts[pair] -= count * ids_count
                if pair_counts[pair] <= 0:
                    del pair_counts[pair]
                pair_to_ids[pair].discard(tuple(ids))

            # 이 코드 단계의 동작을 확인하는 예시
            new_counts = count_pairs(new_ids)
            for pair, count in new_counts.items():
                pair_counts[pair] += count * ids_count
                pair_to_ids[pair].add(tuple(new_ids))

    return merge_rules


#### `BPETokenizer` 클래스 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))

        def get_merge_priority(pair):
            return self.merge_rules.get(pair, float('inf'))  # 이 코드 단계의 동작을 확인하는 예시

        while len(ids) > 1:
            # 이 코드 단계의 동작을 확인하는 예시
            counts = count_pairs(ids)

            # 이 코드 단계의 동작을 확인하는 예시
            best_pair = min(counts, key=get_merge_priority)

            # 이 코드 단계의 동작을 확인하는 예시
            if best_pair not in self.merge_rules:
                break

            # 이 코드 단계의 동작을 확인하는 예시
            new_id = self.merge_rules[best_pair]
            ids = merge(ids, best_pair, new_id)

        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # 이 코드 단계의 동작을 확인하는 예시
        texts = tqdm(texts) if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def _encode_chunk(self, args):
        """チャンクを処理してディスクにキャッシュ"""
        file_path, start, end, cache_dir, chunk_idx = args

        with open(file_path, "rb") as f:
            f.seek(start)
            chunk_byte = f.read(end - start)
            chunk_text = chunk_byte.decode("utf-8", errors="ignore")

            # 이 코드 단계의 동작을 확인하는 예시
            ids = self.encode(chunk_text)

        # 이 코드 단계의 동작을 확인하는 예시
        cache_file = os.path.join(cache_dir, f"chunk_{chunk_idx:05d}.npy")
        np.array(ids, dtype=np.uint16).tofile(cache_file)

        return cache_file, len(ids)


    def encode_file(self, file_path, output_file,
                                    num_processes=4, num_chunks=64,
                                   cache_dir="bpe_cache"):

        # 이 코드 단계의 동작을 확인하는 예시
        os.makedirs(cache_dir, exist_ok=True)

        try:
            # 이 코드 단계의 동작을 확인하는 예시
            chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
            total_chunks = len(chunk_boundaries) - 1

            chunk_info_list = []
            for i in range(total_chunks):
                start = chunk_boundaries[i]
                end = chunk_boundaries[i + 1]
                chunk_info_list.append((file_path, start, end, cache_dir, i))

            with Pool(processes=num_processes) as pool:
                cache_results = list(tqdm(
                    pool.imap(self._encode_chunk, chunk_info_list),
                    total=len(chunk_info_list),
                    desc="Encoding chunks"
                ))

            # 이 코드 단계의 동작을 확인하는 예시
            cache_files = [r[0] for r in cache_results]
            token_counts = [r[1] for r in cache_results]
            total_tokens = sum(token_counts)

            # 이 코드 단계의 동작을 확인하는 예시
            dtype = np.uint16
            arr = np.memmap(output_file, dtype=dtype, mode='w+', shape=(total_tokens,))

            # 이 코드 단계의 동작을 확인하는 예시
            # 이 코드 단계의 동작을 확인하는 예시
            idx = 0
            for cache_file in cache_files:
                chunk_data = np.fromfile(cache_file, dtype=dtype)
                arr[idx : idx + len(chunk_data)] = chunk_data
                idx += len(chunk_data)

            arr.flush()
            del arr

        finally:
            # 이 코드 단계의 동작을 확인하는 예시
            shutil.rmtree(cache_dir)

        return total_tokens

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


### `storybot/utils.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile storybot/utils.py
import torch
import torch.nn.functional as F


#### `generate()` 함수 구현


In [ ]:
%%writefile -a storybot/utils.py


@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=1000, temperature=1.0):
    model.eval()
    model.clear_cache()

    device = next(model.parameters()).device
    ids = tokenizer.encode(prompt)
    ids = torch.tensor([ids], dtype=torch.long, device=device)

    generated_ids = ids
    next_id = ids

    for _ in range(max_new_tokens):
        if ids.size(1) > model.max_context_len:
            ids = ids[:, -model.max_context_len:]

        logits = model(next_id, use_cache=True)[:, -1, :]  # 참고: kv cache
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        if next_id.item() == tokenizer.end_token_id:
            break

        ids = torch.cat((ids, next_id), dim=1)
        generated_ids = torch.cat((generated_ids, next_id), dim=1)

    # 이 코드 단계의 동작을 확인하는 예시
    generated_ids = generated_ids[generated_ids != tokenizer.end_token_id]

    generated_text = tokenizer.decode(generated_ids.tolist())
    return generated_text


#### `get_device()` 함수 구현


In [ ]:
%%writefile -a storybot/utils.py

def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')


## 2. 장별 실습 코드

공식 저장소의 장 코드를 파일 순서대로 모두 다룹니다.


## `ch06/02_adamw.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
from torch.optim.optimizer import Optimizer


### `SGD` 클래스 구현


In [ ]:


class SGD(Optimizer):
    def __init__(self, params, lr=0.01):
        defaults = {'lr': lr}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            lr = group['lr']

            for p in group['params']:
                if p.grad is None:
                    continue

                p.data = p.data - lr * p.grad.data


### `AdamW` 클래스 구현


In [ ]:


class AdamW(Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        defaults = {"lr": lr,
                    "betas": betas,
                    "eps": eps,
                    "weight_decay": weight_decay}
        super().__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            beta1, beta2 = group['betas']

            for p in group['params']:
                if p.grad is None:
                    continue

                grad = p.grad.data
                state = self.state[p]

                if len(state) == 0:
                    state['t'] = 0
                    state['m'] = torch.zeros_like(p.data)
                    state['v'] = torch.zeros_like(p.data)

                state['t'] += 1
                t = state['t']

                # 이 코드 단계의 동작을 확인하는 예시
                m, v = state['m'], state['v']
                m = beta1 * m + (1 - beta1) * grad
                v = beta2 * v + (1 - beta2) * grad**2
                state['m'], state['v'] = m, v

                # 이 코드 단계의 동작을 확인하는 예시
                m_hat = m / (1 - beta1**t)
                v_hat = v / (1 - beta2**t)

                lr, eps, wd = group['lr'], group['eps'], group['weight_decay']
                # 매개변수업데이트
                p.data = p.data - lr * m_hat / (v_hat.sqrt() + eps) - lr * wd * p.data


### 실행 및 결과 확인


In [ ]:


torch.manual_seed(0)


### 설정 및 값 준비: `model`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model = torch.nn.Linear(2, 1)


### 설정 및 값 준비: `optimizer`


In [ ]:
# 참고: optimizer = AdamW(model.parameters(), lr=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1)


### 설정 및 값 준비: `x`


In [ ]:
# 이 코드 단계의 동작을 확인하는 예시
x = torch.tensor([[1.0, 2.0]])


### 설정 및 값 준비: `y`


In [ ]:
y = torch.tensor([[3.0]])


### 반복 실행


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
for step in range(5):
    output = model(x)
    loss = (output - y).pow(2).mean()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"Step {step}: loss = {loss.item():.4f}")


## `ch06/03_lr_scheduler.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### `get_lr()` 함수 구현


In [ ]:
def get_lr(it, max_lr, warmup_iters, max_iters):
    # 워밍업：0 -> max_lr
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    # 이 코드 단계의 동작을 확인하는 예시
    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)

    return 0.0


## `ch06/04_mixed_precision.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch


### 실행 및 결과 확인


In [ ]:

print("----- FP16 -----")


### 설정 및 값 준비: `large`


In [ ]:

large = torch.tensor(1000.0, dtype=torch.float16)


### 설정 및 값 준비: `small`


In [ ]:
small = torch.tensor(0.01, dtype=torch.float16)


### 실행 및 결과 확인


In [ ]:
print(large + small)  # 참고: tensor(1000., dtype=torch.float16)


### 설정 및 값 준비: `tiny`


In [ ]:

tiny = torch.tensor(1e-8, dtype=torch.float16)


### 실행 및 결과 확인


In [ ]:
print(tiny)  # 참고: tensor(0., dtype=torch.float16)


### 설정 및 값 준비: `huge`


In [ ]:

huge = torch.tensor(70000.0, dtype=torch.float16)


### 실행 및 결과 확인


In [ ]:
print(huge)  # 참고: tensor(inf, dtype=torch.float16)


### 실행 및 결과 확인


In [ ]:

print("----- BF16 -----")


### 설정 및 값 준비: `tiny_fp16`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
tiny_fp16 = torch.tensor(1e-8, dtype=torch.float16)


### 실행 및 결과 확인


In [ ]:
print(tiny_fp16)  # 참고: tensor(0., dtype=torch.float16)


### 설정 및 값 준비: `tiny_bf16`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
tiny_bf16 = torch.tensor(1e-8, dtype=torch.bfloat16)


### 실행 및 결과 확인


In [ ]:
print(tiny_bf16)  # 참고: tensor(1.0012e-08, dtype=torch.bfloat16)


### 설정 및 값 준비: `huge_fp16`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
huge_fp16 = torch.tensor(70000.0, dtype=torch.float16)


### 실행 및 결과 확인


In [ ]:
print(huge_fp16)  # 참고: tensor(inf, dtype=torch.float16)


### 설정 및 값 준비: `huge_bf16`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
huge_bf16 = torch.tensor(70000.0, dtype=torch.bfloat16)


### 실행 및 결과 확인


In [ ]:
print(huge_bf16)  # 참고: tensor(70144., dtype=torch.bfloat16)


### 실행 및 결과 확인


In [ ]:


print("----- 自動混合精度 -----")


### 설정 및 값 준비: `device`


In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'


### 설정 및 값 준비: `a`


In [ ]:
a = torch.randn(1000, 1000, device=device)


### 실행 코드


In [ ]:

with torch.autocast(device_type=device, dtype=torch.bfloat16):
    b = a @ a   # 이 코드 단계의 동작을 확인하는 예시
    c = a.sum() # 이 코드 단계의 동작을 확인하는 예시
    print(b.dtype)  # 참고: torch.bfloat16
    print(c.dtype)  # 참고: torch.float32


## `ch06/05_pretrain.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import numpy as np
import torch
import torch.nn.functional as F
from torch.amp import autocast
from tqdm import tqdm
import matplotlib.pyplot as plt
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device


### `get_lr()` 함수 구현


In [ ]:


def get_lr(it, max_lr, warmup_iters, max_iters):
    # 워밍업：0 -> max_lr
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    # 이 코드 단계의 동작을 확인하는 예시
    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)

    return 0.0


### `get_batch()` 함수 구현


In [ ]:


def get_batch(data, context_len, batch_size, device, random=True, offset=0):
    if random:
        ix = torch.randint(len(data) - context_len - 1, (batch_size,))
    else:
        ix = torch.arange(offset, offset + batch_size * context_len, context_len)

        ix = ix[ix + context_len + 1 < len(data)]
        if len(ix) == 0:
            return None, None

    # 배치생성
    x = torch.stack([torch.from_numpy(data[i:i+context_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+context_len+1].astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)


### `evaluate()` 함수 구현


In [ ]:

def evaluate(model, val_data, context_len, batch_size, device):
    """Validation: 全データを順番に処理"""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    max_start = len(val_data) - context_len - 1
    num_batches = (max_start // context_len) // batch_size + 1

    with torch.no_grad():
        for batch_idx in tqdm(range(num_batches), desc="Validation"):
            offset = batch_idx * batch_size * context_len

            x, y = get_batch(val_data, context_len, batch_size, device,
                        random=False, offset=offset)

            if x is None:
                break

            with autocast(device_type=device.type, dtype=torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                    y.view(-1), reduction='sum')

            total_loss += loss.item()
            total_tokens += y.numel()

    model.train()
    return total_loss / total_tokens


### 설정 및 값 준비: `device`


In [ ]:

# 설정
device = get_device()


### 설정 및 값 준비: `data_path`


In [ ]:
data_path = 'storybot/tiny_stories_train.bin'


### 설정 및 값 준비: `val_data_path`


In [ ]:
val_data_path = 'storybot/tiny_stories_valid.bin'


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'storybot/merge_rules.pkl'


### 설정 및 값 준비: `model_save_path`


In [ ]:
model_save_path = 'storybot/model_pretrain.pt'


### 설정 및 값 준비: `context_len`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
context_len = 256


### 설정 및 값 준비: `vocab_size`


In [ ]:
vocab_size = 10000


### 설정 및 값 준비: `batch_size`


In [ ]:
batch_size = 32


### 설정 및 값 준비: `learning_rate`


In [ ]:
learning_rate = 0.001  # 참고: max_lr


### 설정 및 값 준비: `warmup_iters`


In [ ]:
warmup_iters = 200  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `max_iters`


In [ ]:
max_iters = 40000


### 설정 및 값 준비: `embed_dim`


In [ ]:
embed_dim = 512


### 설정 및 값 준비: `n_head`


In [ ]:
n_head = 16


### 설정 및 값 준비: `n_layer`


In [ ]:
n_layer = 4


### 설정 및 값 준비: `ff_dim`


In [ ]:
ff_dim = 1344


### 설정 및 값 준비: `theta`


In [ ]:
theta = 10000


### 설정 및 값 준비: `eval_iters`


In [ ]:
eval_iters = 500


### 설정 및 값 준비: `grad_clip`


In [ ]:
grad_clip = 1.0


### 설정 및 값 준비: `save_iters`


In [ ]:
save_iters = [500, 5000]  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `train_data`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
train_data = np.memmap(data_path, dtype=np.uint16, mode='r')


### 설정 및 값 준비: `val_data`


In [ ]:
val_data = np.memmap(val_data_path, dtype=np.uint16, mode='r')


### 설정 및 값 준비: `tokenizer`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `model`


In [ ]:
model = GPT(
    vocab_size, context_len, embed_dim, n_head, n_layer, ff_dim, theta
).to(device)


### 설정 및 값 준비: `optimizer`


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


### 설정 및 값 준비: `total_params`


In [ ]:

total_params = sum(p.numel() for p in model.parameters())


### 실행 및 결과 확인


In [ ]:
print(f"パラメータ数: {total_params:,} ({total_params/1e6:.1f}M)")


### 설정 및 값 준비: `pbar`


In [ ]:

pbar = tqdm(range(max_iters))


### 설정 및 값 준비: `val_loss`


In [ ]:

val_loss = float('inf')


### 설정 및 값 준비: `val_losses`


In [ ]:
val_losses = []


### 설정 및 값 준비: `val_iters`


In [ ]:
val_iters = []


### 반복 실행


In [ ]:

for i in pbar:
    # 이 코드 단계의 동작을 확인하는 예시
    lr = get_lr(i, learning_rate, warmup_iters, max_iters)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    batch_x, batch_y = get_batch(train_data, context_len, batch_size, device)

    # 이 코드 단계의 동작을 확인하는 예시
    optimizer.zero_grad()

    # 이 코드 단계의 동작을 확인하는 예시
    with autocast(device_type=device.type, dtype=torch.bfloat16):
        logits = model(batch_x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_y.view(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    # 이 코드 단계의 동작을 확인하는 예시
    if i in save_iters:
        save_path = f'storybot/model_iter_{i}.pt'
        model.save(save_path)
        print(f"\nモデルを保存しました（イテレーション {i}）: {save_path}")

    # 이 코드 단계의 동작을 확인하는 예시
    if (i % eval_iters) == 0 or i == max_iters - 1:
        val_loss = evaluate(model, val_data, context_len, batch_size, device)
        val_losses.append(val_loss)
        val_iters.append(i)
    pbar.set_postfix({'loss': f'{loss.item():.4f}', 'val_loss': f'{val_loss:.6f}'})


### 실행 및 결과 확인


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
plt.figure(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:
plt.plot(val_iters, val_losses)


### 실행 및 결과 확인


In [ ]:
plt.xlabel('Iteration')


### 실행 및 결과 확인


In [ ]:
plt.ylabel('Validation Loss')


### 실행 및 결과 확인


In [ ]:
plt.grid(True)


### 실행 및 결과 확인


In [ ]:
plt.savefig('loss_val.png')


### 실행 및 결과 확인


In [ ]:

model.save(model_save_path)


## `ch06/06_generate.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os
import sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate


### 설정 및 값 준비: `device`


In [ ]:

# 설정
device = get_device()


### 설정 및 값 준비: `model_path`


In [ ]:
model_path = 'storybot/model_pretrain.pt'


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'storybot/merge_rules.pkl'


### 설정 및 값 준비: `prompt`


In [ ]:

# 생성설정
# 이 코드 단계의 동작을 확인하는 예시
prompt = "<|endoftext|>"


### 설정 및 값 준비: `max_new_tokens`


In [ ]:
max_new_tokens = 300  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `temperature`


In [ ]:
temperature = 1.0  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `num_samples`


In [ ]:
num_samples = 3  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `tokenizer`


In [ ]:

tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `model`


In [ ]:
model = GPT.load_from(model_path, device=device)


### 반복 실행


In [ ]:

# 텍스트생성
for i in range(num_samples):
    print(f"--- サンプル {i+1} ---")
    story = generate(
        model, tokenizer, prompt, max_new_tokens, temperature
    )
    print(story)


## `ch06/07_llm_judge.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os
import sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import json
import statistics
from openai import OpenAI
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate


### 설정 및 값 준비: `client`


In [ ]:


# 설정
# 참고: ==========================================
client = OpenAI(api_key="your_api_key_here")


### 설정 및 값 준비: `device`


In [ ]:
# 참고: ==========================================
device = get_device()


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'storybot/merge_rules.pkl'


### 설정 및 값 준비: `tokenizer`


In [ ]:
tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `model_paths`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model_paths = {
    500: 'storybot/model_iter_500.pt',
    5000: 'storybot/model_iter_5000.pt',
    40000: 'storybot/model_pretrain.pt',
}


### 설정 및 값 준비: `prompt`


In [ ]:

# 생성설정
prompt = "<|endoftext|>"


### 설정 및 값 준비: `max_new_tokens`


In [ ]:
max_new_tokens = 200


### 설정 및 값 준비: `temperature`


In [ ]:
temperature = 1.0


### 설정 및 값 준비: `num_samples`


In [ ]:
num_samples = 10  # 이 코드 단계의 동작을 확인하는 예시


### `evaluate_story()` 함수 구현


In [ ]:

def evaluate_story(client, story):
    """LLM-as-a-Judgeでストーリーを評価"""

    evaluation_prompt = f"""以下の子供向けストーリーを2つの観点で1-5点で評価してください。

ストーリー:
{story}

評価観点:
1. Coherence（一貫性）: 論理的につながっているか、物語として筋が通っているか
2. Grammar（文法）: 文法的に正しい英語か

以下のJSON形式で回答してください:
{{
    "coherence": <1-5の整数>,
    "grammar": <1-5の整数>,
    "comment": "<評価の簡単な理由>"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": evaluation_prompt}],
        max_tokens=300,
        response_format={"type": "json_object"}
    )

    text = response.choices[0].message.content
    print("=====出力====")
    print(text)

    # 이 코드 단계의 동작을 확인하는 예시
    return json.loads(text)


### 설정 및 값 준비: `results`


In [ ]:

results = {}


### 반복 실행


In [ ]:
for iteration, model_path in model_paths.items():
    print(f"\n{'='*50}")
    print(f"Iteration {iteration}")
    print('='*50)

    model = GPT.load_from(model_path, device=device)
    iteration_results = []

    for i in range(num_samples):
        print(f"\n--- サンプル {i+1} ---")

        # 이 코드 단계의 동작을 확인하는 예시
        story = generate(model, tokenizer, prompt, max_new_tokens, temperature)
        print(f"Story: {story[:200]}...")

        # 이 코드 단계의 동작을 확인하는 예시
        scores = evaluate_story(client, story)
        print(f"Scores: {scores}")

        iteration_results.append({
            "story": story,
            "scores": scores
        })

    results[iteration] = iteration_results


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
print("\n" + "="*50)


### 실행 및 결과 확인


In [ ]:
print("Summary")


### 실행 및 결과 확인


In [ ]:
print("="*50)


### 반복 실행


In [ ]:

for iteration in model_paths.keys():
    scores_list = [r["scores"] for r in results[iteration]]

    print(f"\nIteration {iteration}:")
    for key in ["coherence", "grammar"]:
        values = [s[key] for s in scores_list]
        avg = statistics.mean(values)
        std = statistics.stdev(values) if len(values) > 1 else 0
        print(f"  {key}: {avg:.2f} ± {std:.2f}")


## `ch06/08_dpo.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from itertools import cycle
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device


### 설정 및 값 준비: `device`


In [ ]:

# 설정
device = get_device()


### 설정 및 값 준비: `data_path`


In [ ]:
data_path = 'storybot/tiny_stories_dpo.json'


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'storybot/merge_rules.pkl'


### 설정 및 값 준비: `pretrain_model_path`


In [ ]:
pretrain_model_path = 'storybot/model_pretrain.pt'


### 설정 및 값 준비: `dpo_model_save_path`


In [ ]:
dpo_model_save_path = 'storybot/model_dpo.pt'


### 설정 및 값 준비: `context_len`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
context_len = 256


### 설정 및 값 준비: `batch_size`


In [ ]:
batch_size = 8


### 설정 및 값 준비: `learning_rate`


In [ ]:
learning_rate = 5e-6


### 설정 및 값 준비: `beta`


In [ ]:
beta = 0.1


### 설정 및 값 준비: `max_iters`


In [ ]:
max_iters = 1000


### `DPODataset` 클래스 구현


In [ ]:


class DPODataset(Dataset):
    # 이 코드 단계의 동작을 확인하는 예시
    def __init__(self, data_path, tokenizer, context_len):
        self.tokenizer = tokenizer
        self.context_len = context_len
        self.samples = []

        with open(data_path) as f:
            data = json.load(f)

        for item in data:
            sample = self._create_sample(item['prompt'], item['chosen'], item['rejected'])
            self.samples.append(sample)

    # 이 코드 단계의 동작을 확인하는 예시
    def _pad_and_mask(self, ids, prompt_len):
        mask = [0] * prompt_len + [1] * (len(ids) - prompt_len)

        if len(ids) > self.context_len:
            ids = ids[:self.context_len]
            mask = mask[:self.context_len]
        else:
            pad_len = self.context_len - len(ids)
            ids = ids + [0] * pad_len
            mask = mask + [0] * pad_len

        return ids, mask

    # 이 코드 단계의 동작을 확인하는 예시
    def _create_sample(self, prompt, chosen, rejected):
        prompt_ids = self.tokenizer.encode(prompt)
        chosen_ids = prompt_ids + self.tokenizer.encode(chosen)
        rejected_ids = prompt_ids + self.tokenizer.encode(rejected)

        prompt_len = len(prompt_ids)
        chosen_ids, chosen_mask = self._pad_and_mask(chosen_ids, prompt_len)
        rejected_ids, rejected_mask = self._pad_and_mask(rejected_ids, prompt_len)

        return chosen_ids, chosen_mask, rejected_ids, rejected_mask

    # 이 코드 단계의 동작을 확인하는 예시
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        chosen_ids, chosen_mask, rejected_ids, rejected_mask = self.samples[idx]
        return (
            torch.tensor(chosen_ids, dtype=torch.long),
            torch.tensor(chosen_mask, dtype=torch.long),
            torch.tensor(rejected_ids, dtype=torch.long),
            torch.tensor(rejected_mask, dtype=torch.long),
        )


### `get_sequence_logprobs()` 함수 구현


In [ ]:


def get_sequence_logprobs(model, ids, mask):
    logits = model(ids)  # 출력 예시: (B, C, V)
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)  # 출력 예시: (B, C-1, V)
    labels = ids[:, 1:]  # 출력 예시: (B, C-1)

    per_token_logprobs = torch.gather(
        log_probs, dim=-1, index=labels.unsqueeze(-1)
    ).squeeze(-1)  # 출력 예시: (B, C-1)
    # 이 코드 단계의 동작을 확인하는 예시
    masked_logprobs = per_token_logprobs * mask[:, 1:]
    return masked_logprobs.sum(dim=-1)  # 출력 예시: (B,)


### `compute_dpo_loss()` 함수 구현


In [ ]:


def compute_dpo_loss(model, ref_model, chosen_ids, chosen_mask, rejected_ids, rejected_mask, beta):
    # 이 코드 단계의 동작을 확인하는 예시
    chosen_logprobs = get_sequence_logprobs(model, chosen_ids, chosen_mask)
    rejected_logprobs = get_sequence_logprobs(model, rejected_ids, rejected_mask)

    # 이 코드 단계의 동작을 확인하는 예시
    with torch.no_grad():
        ref_chosen_logprobs = get_sequence_logprobs(ref_model, chosen_ids, chosen_mask)
        ref_rejected_logprobs = get_sequence_logprobs(ref_model, rejected_ids, rejected_mask)

    # 텐서 크기: DPO loss
    logits = beta * (
        (chosen_logprobs - rejected_logprobs) -
        (ref_chosen_logprobs - ref_rejected_logprobs)
    )
    return -F.logsigmoid(logits).mean()


### 설정 및 값 준비: `tokenizer`


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `dataset`


In [ ]:
dataset = DPODataset(data_path, tokenizer, context_len)


### 설정 및 값 준비: `dataloader`


In [ ]:
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


### 설정 및 값 준비: `model`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model = GPT.load_from(pretrain_model_path, device=device)


### 설정 및 값 준비: `ref_model`


In [ ]:
ref_model = GPT.load_from(pretrain_model_path, device=device)


### 실행 및 결과 확인


In [ ]:
ref_model.eval()


### 설정 및 값 준비: `optimizer`


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


### 설정 및 값 준비: `losses`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
losses = []


### 설정 및 값 준비: `data_iter`


In [ ]:
data_iter = cycle(dataloader)


### 설정 및 값 준비: `pbar`


In [ ]:
pbar = tqdm(range(max_iters))


### 반복 실행


In [ ]:

for i in pbar:
    chosen_ids, chosen_mask, rejected_ids, rejected_mask = next(data_iter)
    chosen_ids, chosen_mask = chosen_ids.to(device), chosen_mask.to(device)
    rejected_ids, rejected_mask = rejected_ids.to(device), rejected_mask.to(device)

    loss = compute_dpo_loss(
        model, ref_model,
        chosen_ids, chosen_mask,
        rejected_ids, rejected_mask,
        beta
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
plt.figure(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:
plt.plot(losses)


### 실행 및 결과 확인


In [ ]:
plt.xlabel('Iteration')


### 실행 및 결과 확인


In [ ]:
plt.ylabel('Loss')


### 실행 및 결과 확인


In [ ]:
plt.grid(True)


### 실행 및 결과 확인


In [ ]:
plt.savefig("loss_dpo.png", bbox_inches='tight')


### 실행 및 결과 확인


In [ ]:

model.save(dpo_model_save_path)


## `ch06/09_llm_judge.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os
import sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import json
import torch
import torch.nn.functional as F
from openai import OpenAI
from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate


### 설정 및 값 준비: `client`


In [ ]:

# 설정
# 참고: ==========================================
client = OpenAI(api_key="your_api_key_here")


### 설정 및 값 준비: `device`


In [ ]:
# 참고: ==========================================
device = get_device()


### 설정 및 값 준비: `tokenizer_path`


In [ ]:
tokenizer_path = 'storybot/merge_rules.pkl'


### 설정 및 값 준비: `tokenizer`


In [ ]:
tokenizer = BPETokenizer.load_from(tokenizer_path)


### 설정 및 값 준비: `model_paths`


In [ ]:

# 비교모델
model_paths = {
    'pretrain': 'storybot/model_pretrain.pt',
    'dpo': 'storybot/model_dpo.pt',
}


### 설정 및 값 준비: `prompt`


In [ ]:

# 평가설정
prompt = "Once upon a time"


### 설정 및 값 준비: `num_comparisons`


In [ ]:
num_comparisons = 100  # 비교횟수


### 설정 및 값 준비: `max_new_tokens`


In [ ]:
max_new_tokens = 150


### 설정 및 값 준비: `temperature`


In [ ]:
temperature = 1.0


### `compare_stories()` 함수 구현


In [ ]:


def compare_stories(client, story_a, story_b):
    """2つのストーリーを比較し、どちらがよりハッピーエンドかを判定"""

    evaluation_prompt = f"""以下の2つの子供向けストーリーを比較し、どちらがよりハッピーエンドかを判定してください。

【Story A】
{story_a}

【Story B】
{story_b}

どちらがより明るく幸せな結末か、または希望に満ちた内容かを判断してください。
JSON形式で回答: {{"winner": "A" or "B" or "tie", "reason": "簡潔な理由"}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": evaluation_prompt}],
        max_tokens=150,
        response_format={"type": "json_object"}
    )

    text = response.choices[0].message.content
    return json.loads(text)


### 설정 및 값 준비: `model_pretrain`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model_pretrain = GPT.load_from(model_paths['pretrain'], device=device)


### 설정 및 값 준비: `model_dpo`


In [ ]:
model_dpo = GPT.load_from(model_paths['dpo'], device=device)


### 설정 및 값 준비: `results`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
results = []


### 설정 및 값 준비: `wins`


In [ ]:
wins = {"pretrain": 0, "dpo": 0, "tie": 0}


### 반복 실행


In [ ]:

for i in range(num_comparisons):
    print(f"\n{'='*60}")
    print(f"Comparison {i+1}/{num_comparisons}")
    print('='*60)

    # 이 코드 단계의 동작을 확인하는 예시
    story_pretrain = generate(model_pretrain, tokenizer, prompt, max_new_tokens, temperature)
    story_dpo = generate(model_dpo, tokenizer, prompt, max_new_tokens, temperature)

    print(f"\n[Pretrain]: {story_pretrain[:100]}...")
    print(f"\n[DPO]: {story_dpo[:100]}...")

    # 이 코드 단계의 동작을 확인하는 예시
    import random
    if random.random() < 0.5:
        story_a, story_b = story_pretrain, story_dpo
        mapping = {"A": "pretrain", "B": "dpo"}
    else:
        story_a, story_b = story_dpo, story_pretrain
        mapping = {"A": "dpo", "B": "pretrain"}

    # 이 코드 단계의 동작을 확인하는 예시
    judgment = compare_stories(client, story_a, story_b)

    winner_label = judgment["winner"]
    if winner_label == "tie":
        winner = "tie"
    else:
        winner = mapping[winner_label]

    wins[winner] += 1

    print(f"\n🏆 Winner: {winner}")
    print(f"   Reason: {judgment['reason']}")

    results.append({
        "story_pretrain": story_pretrain,
        "story_dpo": story_dpo,
        "winner": winner,
        "reason": judgment["reason"]
    })


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
print("\n" + "="*60)


### 실행 및 결과 확인


In [ ]:
print("📊 PAIRWISE COMPARISON RESULTS")


### 실행 및 결과 확인


In [ ]:
print("="*60)


### 설정 및 값 준비: `total`


In [ ]:

total = num_comparisons


### 실행 및 결과 확인


In [ ]:
print(f"\n  Pretrain wins: {wins['pretrain']:3d} ({wins['pretrain']/total*100:5.1f}%)")


### 실행 및 결과 확인


In [ ]:
print(f"  DPO wins:      {wins['dpo']:3d} ({wins['dpo']/total*100:5.1f}%)")


### 실행 및 결과 확인


In [ ]:
print(f"  Ties:          {wins['tie']:3d} ({wins['tie']/total*100:5.1f}%)")


### 조건에 따른 실행


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
if wins['pretrain'] + wins['dpo'] > 0:
    dpo_winrate = wins['dpo'] / (wins['pretrain'] + wins['dpo']) * 100
    print(f"\n  DPO win rate (excluding ties): {dpo_winrate:.1f}%")


## `ch06/lr_graph.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


### 실행 코드


In [ ]:

# 폰트 설정
plt.rcParams['font.family'] = 'Hiragino Sans'  # 참고: macOS


### 설정 및 값 준비: `warmup_ratio`


In [ ]:
# plt.rcParams['font.family'] = 'Yu Gothic'  # Windows

# 매개변수
warmup_ratio = 0.05  # 워밍업구간（전체의5%）


### 설정 및 값 준비: `eta_min_ratio`


In [ ]:
eta_min_ratio = 0.1  # 코사인 어닐링의최소 학습률（최대의10%）


### 설정 및 값 준비: `t`


In [ ]:

# 데이터 생성
t = np.linspace(0, 1, 1000)


### `cosine_annealing()` 함수 구현


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
def cosine_annealing(t, warmup_ratio, eta_min_ratio):
    lr = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < warmup_ratio:
            # 워밍업: 0 -> 1
            lr[i] = ti / warmup_ratio
        else:
            # 코사인 어닐링: 1 -> eta_min
            progress = (ti - warmup_ratio) / (1 - warmup_ratio)
            lr[i] = eta_min_ratio + 0.5 * (1 - eta_min_ratio) * (1 + np.cos(np.pi * progress))
    return lr


### `d2z()` 함수 구현


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
def d2z(t, warmup_ratio):
    lr = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < warmup_ratio:
            # 워밍업: 0 -> 1
            lr[i] = ti / warmup_ratio
        else:
            # 선형 감쇠: 1 -> 0
            progress = (ti - warmup_ratio) / (1 - warmup_ratio)
            lr[i] = 1 - progress
    return lr


### 설정 및 값 준비: `cosine_lr`


In [ ]:

cosine_lr = cosine_annealing(t, warmup_ratio, eta_min_ratio)


### 설정 및 값 준비: `d2z_lr`


In [ ]:
d2z_lr = d2z(t, warmup_ratio)


### 실행 코드


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
fig, ax = plt.subplots(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
ax.axvspan(0, warmup_ratio, color='lightgray', alpha=0.5)


### 실행 및 결과 확인


In [ ]:
ax.text(0.01, 1.02, 'ウォームアップ', fontsize=10, va='bottom')


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
ax.plot(t, cosine_lr, 'b-', linewidth=2, label='コサインアニーリング')


### 실행 및 결과 확인


In [ ]:
ax.plot(t, d2z_lr, color='orange', linestyle='--', linewidth=2, label='D2Z')


### 실행 및 결과 확인


In [ ]:

# 축 설정
ax.set_xlim(0, 1)


### 실행 및 결과 확인


In [ ]:
ax.set_ylim(0, 1.05)


### 실행 및 결과 확인


In [ ]:
ax.set_xlabel('学習の進行度', fontsize=12)


### 실행 및 결과 확인


In [ ]:
ax.set_ylabel('学習率', fontsize=12)


### 실행 및 결과 확인


In [ ]:

# 격자
ax.grid(True, linestyle='--', alpha=0.7)


### 실행 및 결과 확인


In [ ]:

# 범례
ax.legend(loc='upper right', fontsize=12)


### 실행 및 결과 확인


In [ ]:

# 여백 조정
plt.tight_layout()


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
plt.savefig('lr_schedule.png', format='png', bbox_inches='tight')


### 실행 및 결과 확인


In [ ]:
plt.close()


## T4 실행 메모

위 코드는 공식 구현의 모델 구조·알고리즘·기본 하이퍼파라미터를 보존합니다. 학습 시간이 긴 셀은 T4에서도 실행 자체는 가능할 수 있지만 전체 스텝 완주에는 시간이 많이 필요할 수 있습니다. 이 노트북은 빠른 실행을 위해 모델을 임의로 축소하거나 핵심 계산을 생략하지 않습니다.
